# Basic Assertion Classification

medspaCy provides flat boolean attributes (`is_negated`, `is_uncertain`, `is_historical`, `is_family`) 
that collapse clinically meaningful distinctions. cwyde adds a richer taxonomy — `AssertionCategory` — 
that preserves the DEFINITE/PROBABLE split and adds INDICATION as a first-class category.

In [1]:
import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])

## medspaCy booleans vs. cwyde categories

The six sentences below illustrate where the two systems agree and where cwyde adds information. 
Key divergences:
- medspaCy collapses `DEFINITE_NEGATED_EXISTENCE` and `PROBABLE_NEGATED_EXISTENCE` into a single `is_negated=True`.
- medspaCy collapses `PROBABLE_EXISTENCE` and `AMBIVALENT_EXISTENCE` into a single `is_uncertain=True`.
- medspaCy misses `INDICATION` entirely — rule-out context is not the same as negation.

In [2]:
sentences = [
    "No evidence of pulmonary embolism.",
    "Probable pulmonary embolism in the right lower lobe.",
    "Cannot exclude pulmonary embolism.",
    "History of pulmonary embolism.",
    "Rule out pulmonary embolism.",
    "Pulmonary embolism is present.",
]

# Header
col_w = [42, 12, 12, 14, 12, 22]
headers = ["Sentence", "is_negated", "is_uncertain", "is_historical", "is_family", "cwyde_category"]
print("".join(h.ljust(col_w[i]) for i, h in enumerate(headers)))
print("-" * sum(col_w))

for sent in sentences:
    doc = nlp(sent)
    for ent in doc.ents:
        row = [
            sent[:40],
            str(ent._.is_negated),
            str(ent._.is_uncertain),
            str(ent._.is_historical),
            str(ent._.is_family),
            ent._.cwyde_assertion_category.value,
        ]
        print("".join(v.ljust(col_w[i]) for i, v in enumerate(row)))

2026-04-28 22:32:24.962 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'No' marked as sentence start (span begin)


2026-04-28 22:32:24.963 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] Token/tag mapping: [(No, True), (evidence, False), (of, False), (pulmonary, False), (embolism, False), (., False)]


2026-04-28 22:32:25.019 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'Probable' marked as sentence start (span begin)


2026-04-28 22:32:25.020 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] Token/tag mapping: [(Probable, True), (pulmonary, False), (embolism, False), (in, False), (the, False), (right, False), (lower, False), (lobe, False), (., False)]


2026-04-28 22:32:25.072 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'Can' marked as sentence start (span begin)


...

2026-04-28 22:32:25.192 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] [doc 0] Token 0 'Rule' marked as sentence start (span begin)


2026-04-28 22:32:25.193 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] Token/tag mapping: [(Rule, True), (out, False), (pulmonary, False), (embolism, False), (., False)]


2026-04-28 22:32:25.255 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 0 'Pulmonary' marked as sentence start (span begin)


2026-04-28 22:32:25.256 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] Token/tag mapping: [(Pulmonary, True), (embolism, False), (is, False), (present, False), (., False)]


History of pulmonary embolism.            False       False       True          False       HISTORICAL            
Rule out pulmonary embolism.              False       False       False         False       INDICATION            
Pulmonary embolism is present.            False       False       False         False       DEFINITE_EXISTENCE    


## Takeaway

- Rows 1 and 3: both show `is_negated=True` in medspaCy, but cwyde distinguishes 
  `DEFINITE_NEGATED_EXISTENCE` (□¬X) from `PROBABLE_NEGATED_EXISTENCE` (◇¬X).
- Row 5 (`Rule out PE`): medspaCy returns `is_negated=False` with no further signal. 
  cwyde returns `INDICATION` — the study is *investigating* PE, not asserting its absence.
- Row 6 (`Pulmonary embolism is present`): baseline `DEFINITE_EXISTENCE` — all booleans False, cwyde confirms.